In [ ]:



suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(data.table)
  library(CellEnergy)
})

if (!requireNamespace("readxl", quietly = TRUE)) {
  install.packages("readxl", repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages({
  library(readxl)
})


expr_path <- "/path/GSE74672/hypoth_moldata_classification08-Mar-2017.xlsx"

out_dir <- "/path/GSE74672/Seurat_CellEnergy_full_pipeline"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

label_col <- "level1_class"

min_cells <- 3
min_features <- 200
n_hvg <- 2000

cat("[INFO] Expression file:\n")
cat(expr_path, "\n\n")

cat("[INFO] Output directory:\n")
cat(out_dir, "\n\n")

if (!file.exists(expr_path)) {
  stop("[ERROR] File does not exist: ", expr_path)
}


trim_string <- function(x) {
  x <- as.character(x)
  x[is.na(x)] <- ""
  gsub("^\\s+|\\s+$", "", x)
}

clean_colname <- function(x) {
  x <- tolower(trim_string(x))
  x <- gsub("[^a-z0-9]+", "_", x)
  x <- gsub("^_+|_+$", "", x)
  x
}

write_matrix_with_id <- function(mat, file, id_col = "ID") {
  df <- as.data.frame(as.matrix(mat), check.names = FALSE)
  df <- cbind(setNames(data.frame(rownames(df), check.names = FALSE), id_col), df)
  data.table::fwrite(df, file = file)
}

minmax_scale_columns <- function(mat) {
  mat <- as.matrix(mat)
  col_min <- apply(mat, 2, min, na.rm = TRUE)
  col_max <- apply(mat, 2, max, na.rm = TRUE)
  col_range <- col_max - col_min
  col_range[col_range == 0 | is.na(col_range)] <- 1

  scaled <- sweep(mat, 2, col_min, FUN = "-")
  scaled <- sweep(scaled, 2, col_range, FUN = "/")
  scaled[is.na(scaled)] <- 0
  scaled
}

get_assay_data_compat <- function(obj, assay = "RNA", layer_name = "counts", slot_name = "counts") {
  DefaultAssay(obj) <- assay
  tryCatch(
    GetAssayData(obj, assay = assay, layer = layer_name),
    error = function(e) GetAssayData(obj, assay = assay, slot = slot_name)
  )
}


cat("[INFO] Reading Excel file...\n")

raw_df <- as.data.frame(
  readxl::read_excel(
    expr_path,
    sheet = 1,
    col_names = FALSE,
    .name_repair = "minimal"
  ),
  check.names = FALSE
)

cat("[INFO] Raw table dimension:\n")
print(dim(raw_df))

if (nrow(raw_df) < 10 || ncol(raw_df) < 10) {
  stop("[ERROR] Input table is too small. Please check the Excel file.")
}

row_key <- trim_string(raw_df[[1]])

cell_ids_all <- trim_string(as.character(raw_df[1, -1, drop = TRUE]))
valid_cell_pos <- which(!is.na(cell_ids_all) & cell_ids_all != "")

if (length(valid_cell_pos) == 0) {
  stop("[ERROR] Cannot detect cell IDs from the first row.")
}

cell_cols <- valid_cell_pos + 1
cell_ids <- make.unique(cell_ids_all[valid_cell_pos])

cat("[INFO] Number of cells detected:\n")
print(length(cell_ids))

cat("[INFO] Example cell IDs:\n")
print(head(cell_ids))


known_meta_rows <- c(
  "level1 class",
  "level2 class (neurons only)",
  "level2 cluster number (neurons only)",
  "age (days postnatal)",
  "sex (female=1,male=-1)",
  "cell diameter",
  "acute stress (true=1)",
  "total molecules"
)

meta_idx <- which(tolower(row_key) %in% tolower(known_meta_rows))

if (length(meta_idx) == 0) {
  stop("[ERROR] Cannot find metadata rows such as level1 class / total molecules.")
}

metadata_mat <- t(as.matrix(raw_df[meta_idx, cell_cols, drop = FALSE]))
colnames(metadata_mat) <- clean_colname(row_key[meta_idx])
rownames(metadata_mat) <- cell_ids

metadata <- as.data.frame(metadata_mat, stringsAsFactors = FALSE, check.names = FALSE)
metadata$Cell <- rownames(metadata)

cat("[INFO] Metadata dimension:\n")
print(dim(metadata))

cat("[INFO] Metadata columns:\n")
print(colnames(metadata))

if (!label_col %in% colnames(metadata)) {
  cat("[ERROR] label_col not found. Available columns are:\n")
  print(colnames(metadata))
  stop("[ERROR] Please check label_col.")
}

metadata[[label_col]] <- trim_string(metadata[[label_col]])
metadata[[label_col]][metadata[[label_col]] == "" | is.na(metadata[[label_col]])] <- NA

cat("[INFO] Label distribution before QC:\n")
print(table(metadata[[label_col]], useNA = "ifany"))

candidate_idx <- setdiff(which(row_key != ""), c(1, meta_idx))

expr_char <- as.matrix(raw_df[candidate_idx, cell_cols, drop = FALSE])

suppressWarnings({
  expr_num <- matrix(
    as.numeric(expr_char),
    nrow = nrow(expr_char),
    ncol = ncol(expr_char)
  )
})

n_numeric <- rowSums(!is.na(expr_num))
keep_gene_row <- n_numeric >= max(3, floor(0.8 * length(cell_ids)))

expr_num <- expr_num[keep_gene_row, , drop = FALSE]
gene_names <- row_key[candidate_idx][keep_gene_row]

valid_gene_name <- !is.na(gene_names) & gene_names != ""
expr_num <- expr_num[valid_gene_name, , drop = FALSE]
gene_names <- gene_names[valid_gene_name]

bad_features <- grepl("^__", gene_names) | gene_names %in% c(
  "no_feature",
  "ambiguous",
  "too_low_aQual",
  "not_aligned",
  "alignment_not_unique",
  "__no_feature",
  "__ambiguous",
  "__too_low_aQual",
  "__not_aligned",
  "__alignment_not_unique"
)

if (any(bad_features)) {
  cat("[INFO] Removing non-gene summary rows:\n")
  print(gene_names[bad_features])
  expr_num <- expr_num[!bad_features, , drop = FALSE]
  gene_names <- gene_names[!bad_features]
}

expr_num[is.na(expr_num)] <- 0

rownames(expr_num) <- gene_names
colnames(expr_num) <- cell_ids

if (any(duplicated(rownames(expr_num)))) {
  cat("[WARN] Duplicated gene names found. Aggregating by sum.\n")
  expr_num <- rowsum(expr_num, group = rownames(expr_num), reorder = FALSE)
}

counts <- as(expr_num, "dgCMatrix")

cat("[INFO] Count matrix dimension, genes x cells:\n")
print(dim(counts))

cat("[INFO] Example genes:\n")
print(head(rownames(counts)))


saveRDS(counts, file.path(out_dir, "full_raw_counts_gene_by_cell.rds"))

data.table::fwrite(
  cbind(Cell = rownames(metadata), metadata[, setdiff(colnames(metadata), "Cell"), drop = FALSE]),
  file.path(out_dir, "metadata_aligned.csv")
)

data.table::fwrite(
  data.frame(Cell = rownames(metadata), Label = metadata[[label_col]], stringsAsFactors = FALSE),
  file.path(out_dir, "cell_labels.csv")
)

cat("[OK] Raw counts and labels saved.\n\n")


seurat_obj <- CreateSeuratObject(
  counts = counts,
  project = "GSE74672",
  meta.data = metadata,
  min.cells = min_cells,
  min.features = min_features
)

cat("[INFO] Seurat object after QC:\n")
print(seurat_obj)

qc_summary <- data.frame(
  n_genes_before_qc = nrow(counts),
  n_cells_before_qc = ncol(counts),
  n_genes_after_qc = nrow(seurat_obj),
  n_cells_after_qc = ncol(seurat_obj),
  min_cells = min_cells,
  min_features = min_features
)

data.table::fwrite(qc_summary, file.path(out_dir, "QC_summary.csv"))

metadata_qc <- seurat_obj@meta.data

cell_label_after_qc <- data.frame(
  Cell = rownames(metadata_qc),
  Label = metadata_qc[[label_col]],
  stringsAsFactors = FALSE
)

data.table::fwrite(
  cell_label_after_qc,
  file.path(out_dir, "cell_labels_after_QC.csv")
)

cat("[INFO] Label distribution after QC:\n")
print(table(cell_label_after_qc$Label, useNA = "ifany"))

seurat_obj <- NormalizeData(
  seurat_obj,
  normalization.method = "LogNormalize",
  scale.factor = 10000
)

n_hvg_use <- min(n_hvg, nrow(seurat_obj))

seurat_obj <- FindVariableFeatures(
  seurat_obj,
  selection.method = "vst",
  nfeatures = n_hvg_use
)

hvg_genes <- VariableFeatures(seurat_obj)

data.table::fwrite(
  data.frame(HVG = hvg_genes),
  file.path(out_dir, paste0("HVG_", length(hvg_genes), "_gene_list.csv"))
)

cat("[INFO] Number of HVGs:\n")
print(length(hvg_genes))

seurat_obj <- ScaleData(
  seurat_obj,
  features = hvg_genes
)


raw_hvg_counts_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "counts",
  slot_name = "counts"
)[hvg_genes, ]

normalized_hvg_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "data",
  slot_name = "data"
)[hvg_genes, ]

scaled_hvg_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "scale.data",
  slot_name = "scale.data"
)[hvg_genes, ]

write_matrix_with_id(
  raw_hvg_counts_gene_by_cell,
  file.path(out_dir, "raw_HVG_counts_gene_by_cell.csv"),
  id_col = "Gene"
)

write_matrix_with_id(
  t(as.matrix(raw_hvg_counts_gene_by_cell)),
  file.path(out_dir, "raw_HVG_counts_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  normalized_hvg_gene_by_cell,
  file.path(out_dir, "normalized_HVG_expr_gene_by_cell.csv"),
  id_col = "Gene"
)

write_matrix_with_id(
  t(as.matrix(normalized_hvg_gene_by_cell)),
  file.path(out_dir, "normalized_HVG_expr_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  t(as.matrix(scaled_hvg_gene_by_cell)),
  file.path(out_dir, "scaled_HVG_expr_cell_by_gene.csv"),
  id_col = "Cell"
)

raw_hvg_cell_by_gene <- t(as.matrix(raw_hvg_counts_gene_by_cell))
raw_hvg_minmax <- minmax_scale_columns(raw_hvg_cell_by_gene)

write_matrix_with_id(
  raw_hvg_minmax,
  file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

cat("[OK] View 1 saved:\n")
cat(file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"), "\n\n")


scaled_for_cellenergy_path <- file.path(out_dir, "scaled_HVG_expr_gene_by_cell_for_CellEnergy.csv")

write.csv(
  as.data.frame(as.matrix(scaled_hvg_gene_by_cell), check.names = FALSE),
  file = scaled_for_cellenergy_path,
  quote = FALSE,
  row.names = TRUE
)

cat("[INFO] Running CellEnergy calcGEn...\n")

if (requireNamespace("reticulate", quietly = TRUE)) {
  if (!reticulate::py_module_available("pandas") ||
      !reticulate::py_module_available("numpy") ||
      !reticulate::py_module_available("scipy")) {
    cat("[WARN] pandas/numpy/scipy may be missing for CellEnergy.\n")
    cat("[WARN] If calcGEn fails, run this in R:\n")
    cat("reticulate::py_install(c('pandas','numpy','scipy'), pip = TRUE)\n")
  }
}

result <- CellEnergy::calcGEn(
  scaled_for_cellenergy_path,
  verbose = TRUE
)

scEn <- result$scEn
GLNE <- result$GLNE

scEn_df <- as.data.frame(scEn, check.names = FALSE)

if (is.null(rownames(scEn_df)) || any(rownames(scEn_df) == "")) {
  rownames(scEn_df) <- colnames(scaled_hvg_gene_by_cell)
}

scEn_df <- cbind(Cell = rownames(scEn_df), scEn_df)

data.table::fwrite(
  scEn_df,
  file.path(out_dir, "Energy_scEn.csv")
)

GLNE_mat <- as.matrix(GLNE)

write_matrix_with_id(
  GLNE_mat,
  file.path(out_dir, "GLNE_raw_output.csv"),
  id_col = "ID"
)

glne_cols_are_cells <- sum(colnames(GLNE_mat) %in% rownames(raw_hvg_cell_by_gene))
glne_rows_are_cells <- sum(rownames(GLNE_mat) %in% rownames(raw_hvg_cell_by_gene))

if (glne_rows_are_cells > glne_cols_are_cells) {
  cat("[INFO] GLNE appears to be cells x genes.\n")
  GLNE_cell_by_gene <- GLNE_mat
} else {
  cat("[INFO] GLNE appears to be genes x cells. Transposing to cells x genes.\n")
  GLNE_cell_by_gene <- t(GLNE_mat)
}

common_cells <- intersect(rownames(raw_hvg_cell_by_gene), rownames(GLNE_cell_by_gene))
common_genes <- intersect(colnames(raw_hvg_cell_by_gene), colnames(GLNE_cell_by_gene))

cat("[INFO] Common cells between View1 and GLNE:\n")
print(length(common_cells))

cat("[INFO] Common genes between View1 and GLNE:\n")
print(length(common_genes))

if (length(common_cells) == 0) {
  stop("[ERROR] No common cells between expression and GLNE.")
}

if (length(common_genes) == 0) {
  stop("[ERROR] No common genes between expression and GLNE.")
}

raw_hvg_cell_by_gene_final <- raw_hvg_cell_by_gene[common_cells, common_genes, drop = FALSE]
GLNE_cell_by_gene <- GLNE_cell_by_gene[common_cells, common_genes, drop = FALSE]

raw_hvg_minmax_final <- minmax_scale_columns(raw_hvg_cell_by_gene_final)
GLNE_minmax <- minmax_scale_columns(GLNE_cell_by_gene)

write_matrix_with_id(
  raw_hvg_cell_by_gene_final,
  file.path(out_dir, "final_raw_HVG_counts_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  GLNE_cell_by_gene,
  file.path(out_dir, "final_GLNE_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  raw_hvg_minmax_final,
  file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  GLNE_minmax,
  file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

cat("[OK] View 2 saved:\n")
cat(file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"), "\n\n")


seurat_rds <- file.path(out_dir, "GSE74672_Seurat_CellEnergy_processed_no_regression.rds")
saveRDS(seurat_obj, file = seurat_rds)

cat("\n[DONE] GSE74672 preprocessing finished.\n")
cat("Important outputs:\n")
cat("1. Labels:\n")
cat(file.path(out_dir, "cell_labels_after_QC.csv"), "\n")
cat("2. View 1:\n")
cat(file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"), "\n")
cat("3. View 2:\n")
cat(file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"), "\n")
cat("4. Seurat RDS:\n")
cat(seurat_rds, "\n")

[INFO] Expression file:
/home/zhanghang/GSE74672/hypoth_moldata_classification08-Mar-2017.xlsx 

[INFO] Output directory:
/home/zhanghang/GSE74672/Seurat_CellEnergy_full_pipeline 

[INFO] Reading Excel file...
[INFO] Raw table dimension:
[1] 24353  2882
[INFO] Number of cells detected:
[1] 2881
[INFO] Example cell IDs:
[1] "1772058147_F02" "1772096158_E08" "1772096144_A05" "1772092004_A05"
[5] "1772092004_B06" "1772096173_F02"
[INFO] Metadata dimension:
[1] 2881    9
[INFO] Metadata columns:
[1] "level1_class"                       "level2_class_neurons_only"         
[3] "level2_cluster_number_neurons_only" "age_days_postnatal"                
[5] "sex_female_1_male_1"                "cell_diameter"                     
[7] "acute_stress_true_1"                "total_molecules"                   
[9] "Cell"                              
[INFO] Label distribution before QC:

 astrocytes endothelial   ependymal   microglia     neurons      oligos 
        267         240         356    

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[INFO] Seurat object after QC:
An object of class Seurat 
18553 features across 2881 samples within 1 assay 
Active assay: RNA (18553 features, 0 variable features)
 1 layer present: counts
[INFO] Label distribution after QC:

 astrocytes endothelial   ependymal   microglia     neurons      oligos 
        267         240         356          48         898        1001 
        vsm 
         71 


Normalizing layer: counts

Finding variable features for layer counts



[INFO] Number of HVGs:
[1] 2000


Centering and scaling data matrix



[OK] View 1 saved:
/home/zhanghang/GSE74672/Seurat_CellEnergy_full_pipeline/final_raw_HVG_counts_MinMax_cell_by_gene.csv 

[INFO] Running CellEnergy calcGEn...


2026-06-11 19:54:04.104446: Starting calculation.

2026-06-11 19:54:47.283711: Complete GLNE and cell Energy calculation.



[INFO] GLNE appears to be genes x cells. Transposing to cells x genes.
[INFO] Common cells between View1 and GLNE:
[1] 2881
[INFO] Common genes between View1 and GLNE:
[1] 2000
[OK] View 2 saved:
/home/zhanghang/GSE74672/Seurat_CellEnergy_full_pipeline/final_GLNE_MinMax_cell_by_gene.csv 


[DONE] GSE74672 preprocessing finished.
Important outputs:
1. Labels:
/home/zhanghang/GSE74672/Seurat_CellEnergy_full_pipeline/cell_labels_after_QC.csv 
2. View 1:
/home/zhanghang/GSE74672/Seurat_CellEnergy_full_pipeline/final_raw_HVG_counts_MinMax_cell_by_gene.csv 
3. View 2:
/home/zhanghang/GSE74672/Seurat_CellEnergy_full_pipeline/final_GLNE_MinMax_cell_by_gene.csv 
4. Seurat RDS:
/home/zhanghang/GSE74672/Seurat_CellEnergy_full_pipeline/GSE74672_Seurat_CellEnergy_processed_no_regression.rds 


In [4]:
library(reticulate)
reticulate::py_install(c("pandas", "numpy", "scipy"), pip = TRUE)

Warning message in reticulate::py_install(c("pandas", "numpy", "scipy"), pip = TRUE):
“An ephemeral virtual environment managed by 'reticulate' is currently in use.
To add more packages to your current session, call `py_require()` instead
of `py_install()`. Running:
  `py_require(c("pandas", "numpy", "scipy"))`”
